## Parameters

In [1]:
xgb_params = {
    'objective': 'multi:softmax', 
    'num_class': 2, 
    'n_estimators': 7, 
    'max_depth': 7, 
    'eta': 0.8
}

lgbm_params = {
    'n_estimators': 7, 
    'max_depth': 7, 
    'eta': 0.8,
    'verbose': 0
}

treelut_params = {
    'w_feature': 3, 
    'w_tree': 3,
    'bits_features': 16,
    'pipeline': [0, 0, 0], 
    'dir_path': './OutputFiles/Example/',
    'style': 'equation',
    'argmax': True,
    'quantized': True}

## MNIST Data Loading

In [2]:
import pandas as pd
from LoadDataset import LoadDataset
from sklearn.model_selection import train_test_split


# adult = pd.read_csv('../misc/data/adult.csv')
# data = adult.drop(columns=['target'])
# target = adult['target'].astype('int')


data, target = LoadDataset.load_adult(debug=True)
data = data.astype(f'uint{treelut_params["bits_features"]}')


X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)

Loaded Adult dataset successfully. Returning data and target variables.
Link to dataset: https://archive.ics.uci.edu/ml/datasets/Adult
Data: (48842, 14)
Target: (48842,)


/home/olavo/Área de trabalho/ufv/projects/TreeLUT/env/lib/python3.13/site-packages/LoadDataset/LoadDataset.py:324: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  adult.data.features[col] = le.fit_transform(adult.data.features[col])
/home/olavo/Área de trabalho/ufv/projects/TreeLUT/env/lib/python3.13/site-packages/LoadDataset/LoadDataset.py:324: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  adult.data.features[col] = le.fit_transform(adult.data.features[col])
/home/olavo/Área de trabalho/ufv/projects/TreeLUT/

In [3]:
# Get number of classes
num_classes = target.nunique()
num_classes



2

In [4]:
data_un = data.astype(f'uint{treelut_params["bits_features"]}')
X_train, X_test, y_train, y_test = train_test_split(data_un, target, test_size=0.2, random_state=42)


# Get max and min of each column of X_test
max_values = X_test.max().astype(f'uint{treelut_params["bits_features"]}')
min_values = X_test.min().astype(f'uint{treelut_params["bits_features"]}')

# To np arrays
max_values = max_values.to_numpy()
min_values = min_values.to_numpy()

print("Max values of each column in X_test:")
print(max_values)
print("Min values of each column in X_test:")
print(min_values)

Max values of each column in X_test:
[   90     9 65535    15    16     6    15     5     4     1 41310  3900
    99    42]
Min values of each column in X_test:
[17  0  9  0  1  0  0  0  0  0  0  0  1  0]


In [5]:
X_train.iloc[:, 5].value_counts()  # Display the first 5 rows and


marital-status
2    17909
4    12914
0     5297
5     1249
6     1188
3      487
1       29
Name: count, dtype: int64

In [6]:
# from torchvision import datasets
# import numpy as np


# data_train = datasets.MNIST(root="./Data", train=True, download=True)
# data_test = datasets.MNIST(root="./Data", train=False, download=True)

# X_train = np.array(data_train.data.numpy()).reshape(data_train.data.shape[0], -1)
# y_train = np.array(data_train.targets.numpy())

# X_test = np.array(data_test.data.numpy()).reshape(data_test.data.shape[0], -1)
# y_test = np.array(data_test.targets.numpy())

## Data Quantization

In [7]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np


scaler = MinMaxScaler()
w_feature = treelut_params['w_feature']
X_train_min_max = np.round(scaler.fit_transform(X_train)*(2**w_feature-1))
X_test_min_max = np.clip(np.round(scaler.transform(X_test)*(2**w_feature-1)), 0, 2**w_feature-1)

# Get the first 4 bits of each feature
w_feature = treelut_params['w_feature']
X_train_quantized = np.array(X_train, dtype=np.int32) & w_feature - 1
X_test_quantized = np.array(X_test, dtype=np.int32) & w_feature - 1 





In [8]:
X_train.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
37193,32,4,50753,11,9,2,14,0,4,1,0,0,40,39
31093,45,7,13279,12,14,2,10,0,4,1,0,0,40,39
33814,35,2,55609,2,8,2,5,0,4,1,0,0,40,39
14500,64,4,3989,11,9,0,3,4,4,1,0,0,20,39
23399,63,6,28612,11,9,6,12,1,4,1,0,0,70,39


In [9]:
# Get the 5 columns of X_train_min_max (a np array)
# Get value count of the column 5 
X_train_min_max = pd.DataFrame(X_train_min_max, columns=X_train.columns)
X_train_min_max.iloc[:, 5].value_counts()  # Display the first 5 rows and


marital-status
2.0    17909
5.0    12914
0.0     5297
6.0     1249
7.0     1188
4.0      487
1.0       29
Name: count, dtype: int64

## XGBoost Model Training

In [10]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

xgb_clf = XGBClassifier(**xgb_params)
xgb_clf.fit(X_train_quantized, y_train)
y_pred_xgb = xgb_clf.predict(X_test_quantized)
print(f"XGB Model Accuracy: {accuracy_score(y_pred_xgb, y_test):.3f}")

XGB Model Accuracy: 0.795


In [11]:
xgb_clf_not_quantized = XGBClassifier(**xgb_params)
xgb_clf_not_quantized.fit(X_train, y_train)
y_pred_xgb_not_quantized = xgb_clf_not_quantized.predict(X_test)
print(f"XGB Model Accuracy (not quantized): {accuracy_score(y_pred_xgb_not_quantized, y_test):.3f}")

XGB Model Accuracy (not quantized): 0.868


In [12]:
xgb_clf_min_max = XGBClassifier(**xgb_params)
xgb_clf_min_max.fit(X_train_min_max, y_train)
y_pred_xgb_min_max = xgb_clf_min_max.predict(X_test_min_max)
print(f"XGB Model Accuracy (min-max scaled): {accuracy_score(y_pred_xgb_min_max, y_test):.3f}")

XGB Model Accuracy (min-max scaled): 0.851


### LightGBM Model Training

In [13]:
from lightgbm import LGBMClassifier

# Treinar modelo LightGBM com dados originais para comparação
lgbm_clf_original = LGBMClassifier(**lgbm_params)
lgbm_clf_original.fit(X_train, y_train)
y_pred_lgbm_original = lgbm_clf_original.predict(X_test)
print(f"LightGBM Model Accuracy (not quantized): {accuracy_score(y_pred_lgbm_original, y_test):.3f}")



# Treinar modelo LightGBM com dados min-max normalizados
lgbm_clf_min_max = LGBMClassifier(**lgbm_params)
lgbm_clf_min_max.fit(X_train_min_max, y_train)
y_pred_lgbm_min_max = lgbm_clf_min_max.predict(X_test_min_max)
print(f"LightGBM Model Accuracy (min-max scaled): {accuracy_score(y_pred_lgbm_min_max, y_test):.3f}")



[LightGBM] [Warning] learning_rate is set=0.1, eta=0.8 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, eta=0.8 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, eta=0.8 will be ignored. Current value: learning_rate=0.1
LightGBM Model Accuracy (not quantized): 0.819
[LightGBM] [Warning] learning_rate is set=0.1, eta=0.8 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, eta=0.8 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, eta=0.8 will be ignored. Current value: learning_rate=0.1
LightGBM Model Accuracy (min-max scaled): 0.811


/home/olavo/Área de trabalho/ufv/projects/TreeLUT/env/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## TreeLUT Model Generation

In [14]:
from treelut import TreeLUTClassifier

# 2025-05-27 13:40:50
# Specifying the parameters for 
treelut_X_test = X_test if treelut_params['quantized'] else X_test_quantized


treelut_clf = TreeLUTClassifier(xgb_model=xgb_clf_min_max, **treelut_params, min=min_values, max=max_values)
treelut_clf.convert()
y_pred_treelut = treelut_clf.predict(X_test_min_max)
print(f"TreeLUT Model Accuracy: {accuracy_score(y_pred_treelut, y_test):.3f}")

treelut_clf.verilog()
treelut_clf.vhdl()
treelut_clf.testbench(treelut_X_test, y_test)


TreeLUT Model Accuracy: 0.848
Info: Generating 19 VHDL modules in ./OutputFiles/Example/TreeLUT/vhdl...
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
input wire
output wire
